In [1]:
import pandas as pd
import numpy as np


In [2]:
# Chargement des secteurs (feuilles du classeur, hors DESCRIPTION)
xlsx_path = "cross_asset_sxxp600.xlsx"
xl = pd.ExcelFile(xlsx_path)
exclude = {"DESCRIPTION", "description", "Desc", "DESC"}
sector_sheets = [s for s in xl.sheet_names if s not in exclude]

def load_sector_sheet(xlsx_path: str, sheet_name: str) -> pd.Series:
    df = pd.read_excel(xlsx_path, sheet_name=sheet_name, skiprows=6)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").set_index("Date")
    return pd.to_numeric(df["PX_LAST"], errors="coerce").rename(sheet_name)

sector_series = [load_sector_sheet(xlsx_path, sh) for sh in sector_sheets]
sectors = pd.concat(sector_series, axis=1).sort_index()
print("sectors:", sectors.shape)


sectors: (6719, 20)


In [3]:
# PHASE 1 - Nettoyage et préparation mensuelle
excluded_tickers = {"S600PDP"}
sectors_clean = sectors.drop(columns=[c for c in excluded_tickers if c in sectors.columns]).copy()

sectors_m = sectors_clean.resample("ME").last().dropna(how="all")
ret_m = sectors_m.pct_change()

bench = pd.read_csv("stoxx600_sectors_history.csv", parse_dates=["Date"]).sort_values("Date").set_index("Date")
bench = bench[["STOXX_600_Global"]].rename(columns={"STOXX_600_Global": "BENCH"})
bench_m = bench.resample("ME").last()
bench_ret_m = bench_m.pct_change()

ret_all_m = ret_m.join(bench_ret_m, how="inner")
sectors_cols = [c for c in ret_all_m.columns if c != "BENCH"]

excess_ret_m = ret_all_m[sectors_cols].sub(ret_all_m["BENCH"], axis=0)
rolling12_excess = (1 + excess_ret_m).rolling(12).apply(np.prod, raw=True) - 1

h = 6
future_ret_6m = (1 + ret_all_m[sectors_cols]).rolling(h).apply(np.prod, raw=True).shift(-h) - 1
future_bench_6m = (1 + ret_all_m["BENCH"]).rolling(h).apply(np.prod, raw=True).shift(-h) - 1
future_excess_6m = future_ret_6m.sub(future_bench_6m, axis=0)

print("ret_all_m:", ret_all_m.shape)
print("future_excess_6m:", future_excess_6m.shape)


ret_all_m: (263, 20)
future_excess_6m: (263, 19)


In [4]:
# PHASE 2 - Regroupement sectoriel en blocs (pondération market cap)
desc = pd.read_excel(
    "cross_asset_sxxp600.xlsx",
    sheet_name="DESCRIPTION",
    header=None,
    usecols="B:E",
    skiprows=1,
    names=["Ticker", "Weight", "Sector", "StartDate"],
)
desc["Weight"] = pd.to_numeric(desc["Weight"], errors="coerce")
desc = desc.dropna(subset=["Ticker", "Weight"])
desc = desc[~desc["Ticker"].isin(excluded_tickers)]
weight_map = desc.set_index("Ticker")["Weight"].to_dict()

macro_blocks = {
    "Cyclical": ["SXNP", "SXOP", "SXPP", "SX4P", "SXAP", "SXTP", "SXRP", "S600ENP"],
    "Defensive": ["SXDP", "SX6P", "S600FOP", "S600CPP", "SXKP", "SXMP"],
    "Financials": ["SX7P", "SXIP", "SXFP"],
    "Tech": ["SX8P"],
    "Real_Estate": ["SX86P"],
}

block_ret_m = pd.DataFrame(index=ret_all_m.index)
for block_name, tickers in macro_blocks.items():
    valid = [t for t in tickers if t in ret_all_m.columns]
    if not valid:
        continue
    w = pd.Series({t: weight_map.get(t, np.nan) for t in valid}).dropna()
    if w.empty:
        w = pd.Series(1.0, index=valid)
    w = w / w.sum()
    block_ret_m[block_name] = ret_all_m[w.index].mul(w, axis=1).sum(axis=1)

block_excess_m = block_ret_m.sub(ret_all_m["BENCH"], axis=0)
block_rolling12_excess = (1 + block_excess_m).rolling(12).apply(np.prod, raw=True) - 1
future_block_ret_6m = (1 + block_ret_m).rolling(h).apply(np.prod, raw=True).shift(-h) - 1
future_block_excess_6m = future_block_ret_6m.sub(future_bench_6m, axis=0)
block_cols = list(block_ret_m.columns)

print("block_ret_m:", block_ret_m.shape)
print("blocks:", block_cols)


block_ret_m: (263, 5)
blocks: ['Cyclical', 'Defensive', 'Financials', 'Tech', 'Real_Estate']


In [5]:
# PHASE 3 - Analyse PMI (lags 0..6) et corrélations blocs, corr avec PMI et surperformance future 6m
pmi_manu = pd.read_csv("pmmneu_m_d.csv", parse_dates=["Date"]).sort_values("Date").set_index("Date")[["Close"]].rename(columns={"Close": "PMI_M"})
pmi_serv = pd.read_csv("pmsreu_m_d.csv", parse_dates=["Date"]).sort_values("Date").set_index("Date")[["Close"]].rename(columns={"Close": "PMI_S"})

pmi = pmi_manu.join(pmi_serv, how="inner")
pmi["PMI"] = pmi.mean(axis=1)
pmi_m = pmi["PMI"].resample("ME").last()

window = 36
pmi_z = (pmi_m - pmi_m.rolling(window).mean()) / pmi_m.rolling(window).std()
pmi_lags = pd.concat({f"lag_{k}": pmi_z.shift(k) for k in range(0, 7)}, axis=1)

records = []
for lag_name in pmi_lags.columns:
    tmp = pd.concat([pmi_lags[lag_name].rename("PMI_LAG"), future_block_excess_6m], axis=1, join="inner").dropna()
    if tmp.empty:
        continue
    corr = tmp[block_cols].corrwith(tmp["PMI_LAG"])
    for b in block_cols:
        records.append({"lag": lag_name, "block": b, "corr": corr.get(b, np.nan), "n_obs": len(tmp)})

corr_pmi_block_lags = pd.DataFrame(records)
corr_matrix = corr_pmi_block_lags.pivot(index="block", columns="lag", values="corr")
corr_matrix = corr_matrix[[f"lag_{k}" for k in range(0, 7) if f"lag_{k}" in corr_matrix.columns]]

idx = corr_pmi_block_lags.groupby("block")["corr"].apply(lambda s: s.abs().idxmax())
best_pmi_lag_by_block = corr_pmi_block_lags.loc[idx].reset_index(drop=True)

def signal_label(x):
    a = abs(x)
    if a >= 0.5:
        return "fort signal (>=50%)"
    if a >= 0.4:
        return "exploitable (>=40%)"
    return "faible (<40%)"

best_pmi_lag_by_block["signal"] = best_pmi_lag_by_block["corr"].apply(signal_label)
best_pmi_lag_by_block = best_pmi_lag_by_block.sort_values("corr", key=lambda s: s.abs(), ascending=False)

print("Corrélations complètes (format long, non filtré):")
print(corr_pmi_block_lags.sort_values(["block", "lag"]).to_string(index=False))
print("\nMatrice complète blocs x lags (non filtrée):")
print(corr_matrix)
print("\nRésumé secondaire (meilleur lag par bloc):")
print(best_pmi_lag_by_block[["block", "lag", "corr", "signal", "n_obs"]])


Corrélations complètes (format long, non filtré):
  lag       block      corr  n_obs
lag_0    Cyclical -0.273288    112
lag_1    Cyclical -0.198515    111
lag_2    Cyclical -0.098529    110
lag_3    Cyclical -0.004204    109
lag_4    Cyclical  0.071485    108
lag_5    Cyclical  0.131235    107
lag_6    Cyclical  0.179396    106
lag_0   Defensive  0.505971    112
lag_1   Defensive  0.456157    111
lag_2   Defensive  0.378619    110
lag_3   Defensive  0.288898    109
lag_4   Defensive  0.241107    108
lag_5   Defensive  0.189889    107
lag_6   Defensive  0.151407    106
lag_0  Financials -0.393077    112
lag_1  Financials -0.406580    111
lag_2  Financials -0.413999    110
lag_3  Financials -0.408951    109
lag_4  Financials -0.438766    108
lag_5  Financials -0.444868    107
lag_6  Financials -0.453365    106
lag_0 Real_Estate  0.136062    112
lag_1 Real_Estate  0.152549    111
lag_2 Real_Estate  0.149948    110
lag_3 Real_Estate  0.102991    109
lag_4 Real_Estate  0.065655    108
lag_5

In [6]:
# Debug tails (aperçu rapide)
print("=== Phase 1 ===")
print("sectors_m tail")
print(sectors_m.tail(3))
print("\nret_all_m tail")
print(ret_all_m.tail(3))
print("\nfuture_excess_6m tail")
print(future_excess_6m.tail(3))

print("\n=== Phase 2 ===")
print("block_ret_m tail")
print(block_ret_m.tail(3))
print("\nfuture_block_excess_6m tail")
print(future_block_excess_6m.tail(3))

print("\n=== Phase 3 ===")
print("pmi_m tail")
print(pmi_m.tail(6))
print("\ncorr_matrix")
print(corr_matrix)
print("\nbest_pmi_lag_by_block")
print(best_pmi_lag_by_block)


print("\nfull corr_pmi_block_lags")
print(corr_pmi_block_lags.sort_values(["block","lag"]).to_string(index=False))


=== Phase 1 ===
sectors_m tail
               SXNP    SX7P     SXDP    SXIP  S600ENP    SX8P  S600FOP  \
Date                                                                     
2025-12-31  1068.30  355.30  1142.11  510.39   134.57  836.60   187.57   
2026-01-31  1133.44  374.61  1179.39  486.52   146.87  880.82   186.15   
2026-02-28  1167.12  369.60  1225.42  498.16   155.23  855.65   200.89   

            S600CPP    SX6P    SXFP    SXOP    SXKP    SXPP     SX4P    SXAP  \
Date                                                                           
2025-12-31   394.90  489.77  899.42  848.04  255.88  666.17  1107.47  524.95   
2026-01-31   364.95  525.38  905.97  852.94  266.64  750.62  1123.25  502.40   
2026-02-28   374.98  542.89  879.25  916.92  299.44  794.28  1195.30  512.03   

             SX86P    SXTP    SXRP    SXMP  
Date                                        
2025-12-31  126.33  280.50  486.28  398.88  
2026-01-31  130.71  270.51  488.34  377.35  
2026-02-28  136.4

Secteur défensif (corr forte et positive): 
Quand PMI est élevé → Defensive surperforme sur les 6 mois suivants.

Secteur financials (corr négative forte): 
PMI élevé → Financials sous-performent 6 mois plus tard.
PMI élevé = souvent fin de cycle
Les marchés anticipent un ralentissement
Les investisseurs se repositionnent vers des valeurs défensives
Financials sont sensibles au cycle et aux taux
Si PMI est au pic, le marché anticipe un ralentissement
Les financières souffrent en phase de décélération

Secteur cyclique (corr faible) : 
Ça veut dire :
Soit le regroupement cyclique est trop large
Soit le PMI n’est pas le bon indicateur pour ce bloc
Soit le marché anticipe déjà tout
Autrement dit :
Le PMI n’apporte pas d’information prédictive exploitable sur ce bloc.


Nb : nOtre shéma claire : PMI → choisit bloc
 Momentum → choisit secteur dans le bloc
 Ici potentiellement : Defensive (signal fort) &
Financials (signal exploitable) 

In [7]:
# PHASE 4 - Signal macro final (adapté aux résultats observés)
# Choix académique: un seul lag global pour éviter l'overfitting.
# Ici on retient lag 0 (le plus robuste via Defensive).
GLOBAL_PMI_LAG = 0

pmi_signal_df = pd.DataFrame(index=pmi_m.index)
pmi_signal_df["PMI_lagged"] = pmi_m.shift(GLOBAL_PMI_LAG)
pmi_signal_df["PMI_Delta_lagged"] = pmi_signal_df["PMI_lagged"].diff()

# Signal macro "PMI fort" vs "PMI faible"
pmi_signal_df["macro_state"] = np.where(
    (pmi_signal_df["PMI_lagged"] > 50) & (pmi_signal_df["PMI_Delta_lagged"] > 0),
    "PMI_fort",
    "PMI_faible",
)

# Traduction allocation (selon vos corrélations):
# PMI fort -> surpondérer Defensive, sous-pondérer Financials
# PMI faible -> réduire Defensive, réintégrer Financials
pmi_signal_df["preferred_block"] = np.where(
    pmi_signal_df["macro_state"] == "PMI_fort", "Defensive", "Financials"
)

# Application en t+1 pour éviter tout look-ahead
pmi_signal_df["preferred_block_trade"] = pmi_signal_df["preferred_block"].shift(1)
pmi_signal_df["macro_state_trade"] = pmi_signal_df["macro_state"].shift(1)

signal_data = pmi_signal_df[["macro_state_trade", "preferred_block_trade"]].join(
    block_ret_m[["Defensive", "Financials"]],
    how="inner",
).dropna()

# Version long-only (switch entre Defensive et Financials)
signal_data["macro_signal_ret"] = np.where(
    signal_data["preferred_block_trade"] == "Defensive",
    signal_data["Defensive"],
    signal_data["Financials"],
)

# Version long/short relative-value (lecture pure du signal)
signal_data["ls_macro_ret"] = np.where(
    signal_data["macro_state_trade"] == "PMI_fort",
    signal_data["Defensive"] - signal_data["Financials"],
    signal_data["Financials"] - signal_data["Defensive"],
)

signal_data["macro_signal_excess_vs_bench"] = signal_data["macro_signal_ret"] - ret_all_m.loc[signal_data.index, "BENCH"]
signal_data["macro_signal_nav"] = 100 * (1 + signal_data["macro_signal_ret"]).cumprod()
signal_data["ls_macro_nav"] = 100 * (1 + signal_data["ls_macro_ret"]).cumprod()

print("GLOBAL_PMI_LAG:", GLOBAL_PMI_LAG)
print("signal_data shape:", signal_data.shape)
print(signal_data[["macro_state_trade", "preferred_block_trade", "Defensive", "Financials", "macro_signal_ret", "ls_macro_ret", "macro_signal_excess_vs_bench"]].tail(12))



GLOBAL_PMI_LAG: 0
signal_data shape: (187, 9)
           macro_state_trade preferred_block_trade  Defensive  Financials  \
Date                                                                        
2025-02-28        PMI_faible            Financials   0.031171    0.088411   
2025-03-31        PMI_faible            Financials  -0.052164   -0.008621   
2025-04-30        PMI_faible            Financials  -0.008546   -0.007198   
2025-05-31        PMI_faible            Financials   0.009545    0.053573   
2025-06-30        PMI_faible            Financials  -0.032254   -0.009202   
2025-07-31        PMI_faible            Financials  -0.013671    0.052373   
2025-08-31          PMI_fort             Defensive   0.022880    0.011341   
2025-09-30          PMI_fort             Defensive  -0.011521    0.022182   
2025-10-31        PMI_faible            Financials   0.043012    0.006104   
2025-11-30          PMI_fort             Defensive   0.031702    0.027944   
2025-12-31          PMI_fort  

In [8]:
# Stats de performance (CAGR, Sharpe, Max DD, IR)
def performance_stats(ret: pd.Series, benchmark_ret: pd.Series | None = None, periods_per_year: int = 12) -> pd.Series:
    r = ret.dropna().copy()
    if len(r) == 0:
        return pd.Series({"CAGR": np.nan, "Sharpe": np.nan, "MaxDD": np.nan, "IR": np.nan})

    nav = (1 + r).cumprod()
    years = len(r) / periods_per_year
    cagr = nav.iloc[-1] ** (1 / years) - 1 if years > 0 else np.nan

    vol = r.std(ddof=1) * np.sqrt(periods_per_year)
    sharpe = (r.mean() * periods_per_year) / vol if vol and not np.isclose(vol, 0.0) else np.nan

    dd = nav / nav.cummax() - 1
    max_dd = dd.min()

    if benchmark_ret is None:
        benchmark_ret = ret_all_m.loc[r.index, "BENCH"]
    br = benchmark_ret.reindex(r.index)
    active = (r - br).dropna()
    if len(active) > 1:
        te = active.std(ddof=1) * np.sqrt(periods_per_year)
        ir = ((active.mean() * periods_per_year) / te) if te and not np.isclose(te, 0.0) else np.nan
    else:
        ir = np.nan

    return pd.Series({"CAGR": cagr, "Sharpe": sharpe, "MaxDD": max_dd, "IR": ir})

print(performance_stats(signal_data["macro_signal_ret"]))
print(performance_stats(signal_data["ls_macro_ret"]))



CAGR      0.083282
Sharpe    0.540439
MaxDD    -0.427725
IR        0.280059
dtype: float64
CAGR      0.029220
Sharpe    0.255756
MaxDD    -0.637699
IR       -0.133815
dtype: float64


In [9]:
# PHASE 5 - Rotation intra-bloc par momentum (mensuel, sans look-ahead)
# Objectif: quand le macro signal active un bloc, sélectionner les secteurs les plus forts dans ce bloc.

TOP_N = 2
MOM_WINDOW = 6

# Momentum 6M calculé sur données passées
mom_6m = (1 + ret_all_m[sectors_cols]).rolling(MOM_WINDOW).apply(np.prod, raw=True) - 1

# Mapping secteurs par bloc (tickers)
active_blocks = {
    "Defensive": ["SXDP", "SX6P", "S600FOP", "S600CPP", "SXKP", "SXMP"],
    "Financials": ["SX7P", "SXIP", "SXFP"],
}

idx = ret_all_m.index
macro_block_trade = pmi_signal_df.reindex(idx)["preferred_block_trade"]

records = []
for i in range(1, len(idx)):
    formation_date = idx[i - 1]   # info dispo à t
    trade_date = idx[i]           # exécution à t+1

    # Momentum seul: top N sur tout l'univers
    mom_all = mom_6m.loc[formation_date].dropna()
    sel_mom = mom_all.nlargest(TOP_N).index.tolist() if len(mom_all) >= TOP_N else []
    ret_mom = ret_all_m.loc[trade_date, sel_mom].mean() if sel_mom else np.nan

    # Macro + Momentum: top N uniquement dans le bloc actif
    block = macro_block_trade.loc[trade_date]
    candidates = [s for s in active_blocks.get(block, []) if s in sectors_cols]
    mom_block = mom_6m.loc[formation_date, candidates].dropna() if len(candidates) else pd.Series(dtype=float)
    sel_macro_mom = mom_block.nlargest(TOP_N).index.tolist() if len(mom_block) >= TOP_N else []
    ret_macro_mom = ret_all_m.loc[trade_date, sel_macro_mom].mean() if sel_macro_mom else np.nan

    records.append(
        {
            "Date": trade_date,
            "macro_block_trade": block,
            "selected_mom_only": ",".join(sel_mom),
            "selected_macro_mom": ",".join(sel_macro_mom),
            "ret_momentum_only": ret_mom,
            "ret_macro_momentum": ret_macro_mom,
        }
    )

phase5_df = pd.DataFrame(records).set_index("Date")

# Ajout benchmark et macro seul pour comparaisons
phase5_df["ret_benchmark"] = ret_all_m.loc[phase5_df.index, "BENCH"]
phase5_df["ret_macro_only"] = signal_data.reindex(phase5_df.index)["macro_signal_ret"]

# NAVs de comparaison
phase5_df["nav_benchmark"] = 100 * (1 + phase5_df["ret_benchmark"].fillna(0)).cumprod()
phase5_df["nav_macro_only"] = 100 * (1 + phase5_df["ret_macro_only"].fillna(0)).cumprod()
phase5_df["nav_momentum_only"] = 100 * (1 + phase5_df["ret_momentum_only"].fillna(0)).cumprod()
phase5_df["nav_macro_momentum"] = 100 * (1 + phase5_df["ret_macro_momentum"].fillna(0)).cumprod()

print("phase5_df shape:", phase5_df.shape)
print(phase5_df[["macro_block_trade", "selected_macro_mom", "ret_macro_momentum", "ret_macro_only", "ret_benchmark"]].tail(12))



phase5_df shape: (262, 11)
           macro_block_trade selected_macro_mom  ret_macro_momentum  \
Date                                                                  
2025-03-31        Financials          SX7P,SXIP            0.013502   
2025-04-30        Financials          SX7P,SXIP           -0.000659   
2025-05-31        Financials          SX7P,SXIP            0.043084   
2025-06-30        Financials          SX7P,SXIP           -0.012177   
2025-07-31        Financials          SX7P,SXIP            0.050421   
2025-08-31         Defensive          SX6P,SXKP            0.002545   
2025-09-30         Defensive          SX6P,SXKP           -0.000817   
2025-10-31        Financials          SX7P,SXIP           -0.001035   
2025-11-30         Defensive       SX6P,S600CPP            0.018872   
2025-12-31         Defensive          SX6P,SXDP            0.009622   
2026-01-31        Financials          SX7P,SXIP            0.003790   
2026-02-28               NaN                      

In [10]:
# Comparaison des performances: Benchmark / Macro seul / Momentum seul / Macro + Momentum
stats_table = pd.DataFrame(
    {
        "Benchmark": performance_stats(phase5_df["ret_benchmark"], benchmark_ret=phase5_df["ret_benchmark"]),
        "Macro_only": performance_stats(phase5_df["ret_macro_only"], benchmark_ret=phase5_df["ret_benchmark"]),
        "Momentum_only": performance_stats(phase5_df["ret_momentum_only"], benchmark_ret=phase5_df["ret_benchmark"]),
        "Macro_plus_Momentum": performance_stats(phase5_df["ret_macro_momentum"], benchmark_ret=phase5_df["ret_benchmark"]),
    }
).T

print(stats_table)



                         CAGR    Sharpe     MaxDD        IR
Benchmark            0.044499  0.383185 -0.564148       NaN
Macro_only           0.083282  0.540439 -0.427725  0.280059
Momentum_only        0.061796  0.445121 -0.580678  0.188691
Macro_plus_Momentum  0.082146  0.574992 -0.358129  0.303305


Analyse factorielle : Clustering des sensibilités pour phase 6

In [11]:
# PHASE 6 - Sensibilités factorielles (PMI / Taux / Inflation breakeven)
cross = pd.read_excel("cross_asset.xlsx", header=None)

def find_cell(df: pd.DataFrame, text: str):
    m = df.astype(str).apply(lambda col: col.str.contains(text, case=False, na=False))
    if not m.any().any():
        return None
    r, c = np.argwhere(m.values)[0]
    return int(r), int(c)

def extract_series_under_label(df: pd.DataFrame, label_candidates: list[str], min_date: str = "2000-01-01") -> pd.Series:
    pos = None
    for lb in label_candidates:
        pos = find_cell(df, lb)
        if pos is not None:
            break
    if pos is None:
        raise ValueError(f"Label introuvable parmi: {label_candidates}")

    r, c = pos
    dates = pd.to_datetime(df.iloc[r+1:, c], errors="coerce")
    values = pd.to_numeric(df.iloc[r+1:, c+1], errors="coerce")
    s = pd.Series(values.values, index=dates).dropna()
    s = s[~s.index.duplicated(keep="first")].sort_index()
    s = s[s.index >= pd.Timestamp(min_date)]
    return s

# Facteurs mensuels
factor_pmi_m = pmi_m.copy()
factor_rates_m = extract_series_under_label(cross, ["10Y Bund nominal"]).resample("ME").last()
factor_breakeven_m = extract_series_under_label(cross, ["DEGGBE10 Index", "BREAKEVEN"]).resample("ME").last()

factor_map = {
    "PMI": factor_pmi_m,
    "Rates_10Y": factor_rates_m,
    "Breakeven_10Y": factor_breakeven_m,
}

def compute_sector_sensitivity(factor_m: pd.Series, factor_name: str, target_df: pd.DataFrame, lag_max: int = 6):
    rows = []
    for sec in target_df.columns:
        best = {"sector": sec, "factor": factor_name, "best_lag": np.nan, "best_corr": np.nan, "n_obs": 0}
        best_abs = -1
        for lag in range(lag_max + 1):
            f_lag = factor_m.shift(lag).rename("factor")
            tmp = pd.concat([f_lag, target_df[sec].rename("y")], axis=1, join="inner").dropna()
            if len(tmp) < 24:
                continue
            corr = tmp["factor"].corr(tmp["y"])
            if pd.isna(corr):
                continue
            if abs(corr) > best_abs:
                best_abs = abs(corr)
                best = {
                    "sector": sec,
                    "factor": factor_name,
                    "best_lag": lag,
                    "best_corr": corr,
                    "n_obs": len(tmp),
                }
        rows.append(best)
    return pd.DataFrame(rows)

sens_long = []
for fname, fseries in factor_map.items():
    sens_long.append(compute_sector_sensitivity(fseries, fname, future_excess_6m, lag_max=6))
sens_long = pd.concat(sens_long, ignore_index=True)

print("sens_long shape:", sens_long.shape)
print(sens_long.head())



sens_long shape: (57, 5)
    sector factor  best_lag  best_corr  n_obs
0     SXNP    PMI         0  -0.470867    182
1     SX7P    PMI         5  -0.448108    177
2     SXDP    PMI         5   0.300200    177
3     SXIP    PMI         6  -0.181351    176
4  S600ENP    PMI         0   0.475283    179


In [12]:
# Matrice de sensibilités + clustering (KMeans numpy, sans dépendance externe)
def standardize_df(df: pd.DataFrame) -> pd.DataFrame:
    mu = df.mean(axis=0)
    sigma = df.std(axis=0, ddof=0).replace(0, np.nan)
    return (df - mu) / sigma

def kmeans_numpy(X: np.ndarray, n_clusters: int = 3, random_state: int = 42, n_init: int = 20, max_iter: int = 300):
    rng = np.random.default_rng(random_state)
    best_inertia = np.inf
    best_labels = None
    best_centers = None

    for _ in range(n_init):
        init_idx = rng.choice(len(X), size=n_clusters, replace=False)
        centers = X[init_idx].copy()

        for _ in range(max_iter):
            d2 = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
            labels = d2.argmin(axis=1)

            new_centers = centers.copy()
            for k in range(n_clusters):
                members = X[labels == k]
                if len(members) > 0:
                    new_centers[k] = members.mean(axis=0)
            if np.allclose(new_centers, centers, atol=1e-8):
                centers = new_centers
                break
            centers = new_centers

        inertia = ((X - centers[labels]) ** 2).sum()
        if inertia < best_inertia:
            best_inertia = inertia
            best_labels = labels.copy()
            best_centers = centers.copy()

    return best_labels, best_centers, best_inertia

beta_mat = sens_long.pivot(index="sector", columns="factor", values="best_corr")
lag_mat = sens_long.pivot(index="sector", columns="factor", values="best_lag")

try:
    ticker_to_name = desc.set_index("Ticker")["Sector"].to_dict()
except Exception:
    ticker_to_name = {}

sensitivity_matrix = beta_mat.rename(columns={
    "PMI": "Beta_PMI",
    "Rates_10Y": "Beta_Rates",
    "Breakeven_10Y": "Beta_Inflation",
}).copy()
sensitivity_matrix["SectorName"] = [ticker_to_name.get(t, t) for t in sensitivity_matrix.index]
sensitivity_matrix["BestLag_PMI"] = lag_mat.get("PMI")
sensitivity_matrix["BestLag_Rates"] = lag_mat.get("Rates_10Y")
sensitivity_matrix["BestLag_Inflation"] = lag_mat.get("Breakeven_10Y")

feature_cols = ["Beta_PMI", "Beta_Rates", "Beta_Inflation"]
X = sensitivity_matrix[feature_cols].copy()
valid_idx = X.dropna().index
X_scaled = standardize_df(X.loc[valid_idx]).values

labels, centers, inertia = kmeans_numpy(X_scaled, n_clusters=3, random_state=42, n_init=20)

sensitivity_matrix["cluster"] = np.nan
sensitivity_matrix.loc[valid_idx, "cluster"] = labels
sensitivity_matrix["cluster"] = sensitivity_matrix["cluster"].astype("Int64")

print("Matrice de sensibilités (entrée clustering):")
print(sensitivity_matrix.sort_values(["cluster", "SectorName"]).to_string())

cluster_profile = sensitivity_matrix.groupby("cluster")[feature_cols].mean()
print("\nProfil moyen par cluster:")
print(cluster_profile)
print("\nInertia:", inertia)



Matrice de sensibilités (entrée clustering):
factor   Beta_Inflation  Beta_PMI  Beta_Rates                                                 SectorName  BestLag_PMI  BestLag_Rates  BestLag_Inflation  cluster
sector                                                                                                                                                          
SXAP          -0.163392 -0.255602   -0.099283             STOXX Europe 600 Automobiles & Parts Price EUR            3              6                  0        0
SXPP          -0.297092  0.135923   -0.232616                 STOXX Europe 600 Basic Resources Price EUR            1              6                  0        0
SX4P          -0.296009  0.225856   -0.176041                       STOXX Europe 600 Chemicals Price EUR            6              0                  0        0
SXOP          -0.178392 -0.331893   -0.120773        STOXX Europe 600 Construction & Materials Price EUR            0              0                  

Le clustering basé sur les sensibilités factorielles révèle trois thématiques macro distinctes.

Cluster 0 : secteurs sensibles aux conditions financières, pénalisés par la hausse des taux et de l’inflation 
Cluster 1 : secteurs bénéficiant des hausses de taux et relativement mieux positionnés en phase de ralentissement 
Cluster 2 : secteurs fortement exposés au cycle d’activité, avec une sensibilité positive marquée au PMI et faible dépendance aux taux et à l’inflation 
Conclusion : les regroupements sectoriels dépendent du facteur macro étudié ; le découpage traditionnel cyclique/défensif ne capture pas entièrement ces dynamiques factorielles.

La suite à faire selon moi : intégrer plusieurs facteurs macro (cycle, taux, inflation) dans un score agrégé (exemple : Signal 1 : Cycle (PMI)
PMI > 50 & ΔPMI > 0 → +1
sinon → -1
Signal 2 : Taux
Taux en hausse → favorable aux Banks
Taux en baisse → favorable Real Estate / Utilities
Signal 3 : Inflation
Breakeven en hausse → favorable Energy / Materials )